In [40]:
import pandas as pd
import numpy as np
import ustreasurycurve as ustcurve
from datetime import datetime
from zoneinfo import ZoneInfo
from io import StringIO

# Today's date in Czech time
today = pd.Timestamp(datetime.now(ZoneInfo("Europe/Prague")).date())
start = today - pd.Timedelta(days=37)

In [41]:
# Get Treasury data
df = ustcurve.nominal_rates(
    start.strftime("%Y-%m-%d"),
    today.strftime("%Y-%m-%d")
)[['date', '2y', '10y']]

df['date'] = pd.to_datetime(df['date'])

# Business days for the last 31 days
dates = pd.DataFrame({
    'date': pd.date_range(start, today, freq='B')
})

# Match each business day to the latest available Treasury observation
df_usd = pd.merge_asof(
    dates,
    df.assign(source_date=df['date']).sort_values('date'),
    on='date',
    direction='backward'
)

# One-observation lag
df_usd[['2y', '10y']] = df_usd[['2y', '10y']].shift(1)

df_usd.rename(columns={'2y': '2Y', '10y': '10Y'}, inplace=True)

# Show the result
df_usd

,date,2Y,10Y,source_date
0,2026-08-31,NaN,NaN,2026-08-31
1,2026-09-01,4.34,4.75,2026-09-01
2,2026-09-02,4.39,4.79,2026-09-02
3,2026-09-03,4.39,4.79,2026-09-03
4,2026-09-04,4.34,4.77,2026-09-04
5,2026-09-07,4.37,4.78,2026-09-04
6,2026-09-08,4.37,4.78,2026-09-08
7,2026-09-09,4.39,4.80,2026-09-09
8,2026-09-10,4.43,4.83,2026-09-10
9,2026-09-11,4.56,4.95,2026-09-11


In [42]:
# Download ECB AAA yield curve data
url = (
    "https://data-api.ecb.europa.eu/service/data/"
    "YC/B.U2.EUR.4F.G_N_A.SV_C_YM.SR_2Y+SR_10Y"
)

response = requests.get(
    url,
    params={
        "startPeriod": start.strftime("%Y-%m-%d"),
        "endPeriod": today.strftime("%Y-%m-%d"),
        "format": "csvdata"
    },
    timeout=60
)
response.raise_for_status()

df_eur = pd.read_csv(StringIO(response.text))

# Keep the two maturities and reshape to columns
df_eur = (
    df_eur[df_eur["DATA_TYPE_FM"].isin(["SR_2Y", "SR_10Y"])]
    .pivot(index="TIME_PERIOD", columns="DATA_TYPE_FM", values="OBS_VALUE")
    .rename(columns={"SR_2Y": "2Y", "SR_10Y": "10Y"})
)

df_eur.columns.name = None
df_eur = df_eur[["2Y", "10Y"]]

df_eur.index = pd.to_datetime(df_eur.index)
df_eur = df_eur.apply(pd.to_numeric, errors="coerce").sort_index()

# Create business-day calendar and carry forward ECB holidays
df_eur = df_eur.reindex(
    pd.date_range(start, today, freq="B")
)
df_eur.index.name = "date"

# Keep track of the actual ECB observation date
df_eur["source_date"] = pd.Series(
    df_eur.index.where(df_eur["2Y"].notna()),
    index=df_eur.index
).ffill()

df_eur[["2Y", "10Y"]] = df_eur[["2Y", "10Y"]].ffill()

# Apply the same one-row lag as the US data
df_eur[["2Y", "10Y", "source_date"]] = (df_eur[["2Y", "10Y", "source_date"]].shift(1))

df_eur = df_eur.reset_index()

df_eur = df_eur[["date", "2Y", "10Y", "source_date"]]

df_eur

,date,2Y,10Y,source_date
0,2026-08-31,NaN,NaN,NaT
1,2026-09-01,2.856018,3.339594,2026-08-31
2,2026-09-02,2.876001,3.355152,2026-09-01
3,2026-09-03,2.913206,3.391851,2026-09-02
4,2026-09-04,2.885664,3.364624,2026-09-03
5,2026-09-07,2.876637,3.346497,2026-09-04
6,2026-09-08,2.913307,3.389493,2026-09-07
7,2026-09-09,2.919754,3.378456,2026-09-08
8,2026-09-10,2.976293,3.426566,2026-09-09
9,2026-09-11,3.100596,3.503209,2026-09-10


In [44]:
# 1. Keep the relevant columns and rename them clearly

usd = df_usd[['date', '2Y', '10Y']].copy()
eur = df_eur[['date', '2Y', '10Y']].copy()

usd = usd.rename(columns={
    '2Y': 'US_2Y',
    '10Y': 'US_10Y'
})

eur = eur.rename(columns={
    '2Y': 'EUR_2Y',
    '10Y': 'EUR_10Y'
})


# 2. Make sure dates have the same data type

usd['date'] = pd.to_datetime(usd['date'])
eur['date'] = pd.to_datetime(eur['date'])


# 3. Combine both yield curves into one table

df_combined = pd.merge(
    usd,
    eur,
    on='date',
    how='outer'
).sort_values('date').reset_index(drop=True)


# 4. Calculate CURRENT yield differentials (US minus EUR)

df_combined['US_EUR_2Y_Diff'] = (
    df_combined['US_2Y'] - df_combined['EUR_2Y']
)

df_combined['US_EUR_10Y_Diff'] = (
    df_combined['US_10Y'] - df_combined['EUR_10Y']
)


# 5. Calculate the change in the differential
#    versus the SAME WEEKDAY one week ago
#
#    Because weekends are excluded:
#    shift(5) = previous week's same weekday
#
#    Example:
#    Monday Sep 14 -> Monday Sep 7
#    Tuesday Sep 15 -> Tuesday Sep 8
#    Wednesday Sep 16 -> Wednesday Sep 9

df_combined['US_EUR_2Y_1W_Change'] = (
    df_combined['US_EUR_2Y_Diff']
    - df_combined['US_EUR_2Y_Diff'].shift(5)
)

df_combined['US_EUR_10Y_1W_Change'] = (
    df_combined['US_EUR_10Y_Diff']
    - df_combined['US_EUR_10Y_Diff'].shift(5)
)


# 6. Generate the 2Y trading signal
#
#    IMPORTANT:
#    The signal is based on the 1-week CHANGE,
#    NOT on the current US_EUR_2Y_Diff.

df_combined['Signal_2Y'] = np.select(
    [
        df_combined['US_EUR_2Y_1W_Change'] > 0.05,
        df_combined['US_EUR_2Y_1W_Change'] < -0.05
    ],
    [
        'SELL EUR/USD 2Y',
        'BUY EUR/USD 2Y'
    ],
    default='NEUTRAL EUR/USD 2Y'
)


# 7. Generate the 10Y trading signal
#
#    Again, based on the 1-week CHANGE.

df_combined['Signal_10Y'] = np.select(
    [
        df_combined['US_EUR_10Y_1W_Change'] > 0.05,
        df_combined['US_EUR_10Y_1W_Change'] < -0.05
    ],
    [
        'SELL EUR/USD 10Y',
        'BUY EUR/USD 10Y'
    ],
    default='NEUTRAL EUR/USD 10Y'
)


# 8. Remove rows where the 1-week changes are NaN

df_combined = df_combined.dropna(
    subset=[
        'US_EUR_2Y_1W_Change',
        'US_EUR_10Y_1W_Change'
    ]
)


# 9. Round numerical columns

df_combined = df_combined.round(4)


# 10. Display the final table

display(df_combined)

C:\Users\Vladyslav\AppData\Local\Temp\ipykernel_6292\4261142388.py:114: UserWarning: obj.round has no effect with datetime, timedelta, or period dtypes. Use obj.dt.round(...) instead.
  df_combined = df_combined.round(4)


,date,US_2Y,US_10Y,EUR_2Y,EUR_10Y,US_EUR_2Y_Diff,US_EUR_10Y_Diff,US_EUR_2Y_1W_Change,US_EUR_10Y_1W_Change,Signal_2Y,Signal_10Y
6,2026-09-08,4.37,4.78,2.9133,3.3895,1.4567,1.3905,-0.0273,-0.0199,NEUTRAL EUR/USD 2Y,NEUTRAL EUR/USD 10Y
7,2026-09-09,4.39,4.80,2.9198,3.3785,1.4702,1.4215,-0.0438,-0.0133,NEUTRAL EUR/USD 2Y,NEUTRAL EUR/USD 10Y
8,2026-09-10,4.43,4.83,2.9763,3.4266,1.4537,1.4034,-0.0231,0.0053,NEUTRAL EUR/USD 2Y,NEUTRAL EUR/USD 10Y
9,2026-09-11,4.56,4.95,3.1006,3.5032,1.4594,1.4468,0.0051,0.0414,NEUTRAL EUR/USD 2Y,NEUTRAL EUR/USD 10Y
10,2026-09-14,4.63,4.96,3.1276,3.5227,1.5024,1.4373,0.0091,0.0038,NEUTRAL EUR/USD 2Y,NEUTRAL EUR/USD 10Y
11,2026-09-15,4.65,4.97,3.2278,3.5409,1.4222,1.4291,-0.0345,0.0386,NEUTRAL EUR/USD 2Y,NEUTRAL EUR/USD 10Y
12,2026-09-16,4.67,5.00,3.1878,3.5437,1.4822,1.4563,0.0119,0.0348,NEUTRAL EUR/USD 2Y,NEUTRAL EUR/USD 10Y
13,2026-09-17,4.74,5.01,3.1658,3.5335,1.5742,1.4765,0.1205,0.0731,SELL EUR/USD 2Y,SELL EUR/USD 10Y
14,2026-09-18,4.67,4.94,3.1482,3.4875,1.5218,1.4525,0.0624,0.0057,SELL EUR/USD 2Y,NEUTRAL EUR/USD 10Y
15,2026-09-21,4.76,5.01,3.2152,3.5298,1.5448,1.4802,0.0424,0.0430,NEUTRAL EUR/USD 2Y,NEUTRAL EUR/USD 10Y
